### Individual Conditional Expectation (ICE)

インスタンスごとの異質性を捉える手法

ex. 飲食店で料理のボリュームを増やしてみると

若い人は喜ぶ but 中高年が喜ぶとは思えない

学習済みモデル $\hat{f} ({\bf X})$

インスタンス$i$の$j$番目の特徴量を除いたベクトル ${\bf x}_{i, \backslash j} = (x_{i, 1}, \ldots, x_{i, j-1}, x_{i, j+1}, \ldots, x_{i, J})$

$\widehat{ICE}_{i, j} (x_j) = \hat {f} (x_j, {\bf x}_{i, \backslash j})$

#### 交互作用がある場合のPD

シミュレーションモデル1

$Y = X_0 - 5X_1 + 10X_1X_2 + \varepsilon$

$X_0 \sim {\rm U} (-1, 1)$

$X_1 \sim {\rm U} (-1, 1)$

$X_2 \sim {\rm Bernoulli} (0.5)$

$\varepsilon \sim {\mathcal N} (0, 0.01)$

$X_2$は0と1の値を半々の確率で取る

$X_2 = 0$のときは$X_1$は$Y$に対して負の影響 $X_2 = 1$のときは正の影響

In [ ]:
# シミュレーション

import sys
import warnings
from dataclasses import dataclass
from typing import Any  # 型ヒント用
from __future__ import annotations  # 型ヒント用

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib  # matplotlibの日本語表示対応

# 自作モジュール

from mli.visualize import get_visualization_setting

np.random.seed(42)
pd.options.display.float_format = "{:.2f}".format
sns.set(**get_visualization_setting())
warnings.simplefilter("ignore")  # warningsを非表示に

from sklearn.model_selection import train_test_split

def generate_simulation_data():
    """シミュレーションデータを生成し、訓練データとテストデータに分割"""
    
    # シミュレーションの設定
    N = 1000
    
    # X0とX1は一様分布から生成
    x0 = np.random.uniform(-1, 1, N)
    x1 = np.random.uniform(-1, 1, N)
    # 二項分布の試行回数を1にすると成功確率0.5のベルヌーイ分布と一致
    x2 = np.random.binomial(1, 0.5, N)
    # ノイズは正規分布からデータを生成
    epsilon = np.random.normal(0, 0.1, N)
    
    # 特徴量をまとめる
    X = np.column_stack((x0, x1, x2))
    
    # 線形和で目的変数を作成
    y = x0 - 5 * x1 + 10 * x1 * x2 + epsilon

    return train_test_split(X, y, test_size=0.2, random_state=42)


# シミュレーションデータを生成

X_train, X_test, y_train, y_test = generate_simulation_data()

def plot_scatter(x, y, title=None, xlabel=None, ylabel=None):
    """散布図を作成する"""
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.3)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    fig.suptitle(title)

    fig.show()


# 散布図を可視化

plot_scatter(x=X_train[:, 1], y=y_train, title="X1とYの散布図", xlabel="X1", ylabel="Y")

# 見事なX型

In [ ]:
# ランダムフォレストで予測モデルを構築

from sklearn.ensemble import RandomForestRegressor
from mli.metrics import regression_metrics

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# 予測精度を確認

regression_metrics(rf, X_test, y_test)

# 非常にまともな決定係数

In [ ]:
# X1についてPDを計算して可視化

from mli.interpret import PartialDependence

pdp = PartialDependence(rf, X_test, ["X0", "X1", "X2"])
pdp.partial_dependence("X1")
pdp.plot(ylim=(-6, 6))

# X1の影響がほとんどない!?

$X_1$の影響をPDで検出できない理由

$\hat{f} (X_0, X_1, X_2) = X_0 - 5X_1 + 10X_1X_2$だったから

${\rm PD}_1(x_1) = \mathbb{E} [\hat{f} (X_0, x_1, X_2)] = \mathbb{E} [X_0 - 5x_1 + 10x_1X_2] = \mathbb{E} [X_0] - 5x_1 +10x_1 \mathbb{E}[X_2] = 0 - 5x_1 + 10x_1 \times 0.5 = 0$

ICEだとどうなるか??

${\rm ICE}_{i, 1} (x_1) = \hat{f} (x_{i, 0}, x_1, x_{i, 2}) = x_{i, 0} - 5x_1 + 10x_1x_{i, 2}$

$x_{i, 2} = 0$のときは$x_{i, 0} - 5x_1$ $x_{i, 2} = 1$のときは$x_{i, 0} + 5x_1$

PDではうまくいかない交互作用もうまく捉えている

In [ ]:
# お手製のICEのメソッドを試す

class IndividualConditionalExpectation(PartialDependence):
    """Individual Conditional Expectation"""

    def individual_conditional_expectation(
        self, 
        var_name: str, 
        ids_to_compute: list[int], 
        n_grid: int = 50
    ) -> None:
        """ICEを求める

        Args:
            var_name:
                ICEを計算したい変数名
            ids_to_compute:
                ICEを計算したいインスタンスのリスト
            n_grid: 
                グリッドを何分割するか
                細かすぎると値が荒れるが、粗すぎるとうまく関係をとらえられない
                デフォルトは50
        """
        
        # 可視化の際に用いるのでターゲットの変数名を保存
        self.target_var_name = var_name 
        # 変数名に対応するインデックスをもってくる
        var_index = self.var_names.index(var_name)

        # ターゲットの変数を、取りうる値の最大値から最小値まで動かせるようにする
        value_range = np.linspace(
            self.X[:, var_index].min(),
            self.X[:, var_index].max(),
            num=n_grid
        )

        # インスタンスごとのモデルの予測値
        # PDの_counterfactual_prediction()をそのまま使っているので
        # 全データに対して予測してからids_to_computeに絞り込んでいるが
        # 本当は絞り込んでから予測をしたほうが速い
        individual_prediction = np.array([
            self._counterfactual_prediction(var_index, x)[ids_to_compute]
            for x in value_range
        ])

        # ICEをデータフレームとしてまとめる
        self.df_ice = (
            # ICEの値
            pd.DataFrame(data=individual_prediction, columns=ids_to_compute)
            # ICEで用いた特徴量の値。特徴量名を列名としている
            .assign(**{var_name: value_range})
            # 縦持ちに変換して完成
            .melt(id_vars=var_name, var_name="instance", value_name="ice")
        )

        # ICEを計算したインスタンスについての情報も保存しておく
        # 可視化の際に実際の特徴量の値とその予測値をプロットするために用いる
        self.df_instance = (
            # インスタンスの特徴量の値
            pd.DataFrame(
                data=self.X[ids_to_compute],
                columns=self.var_names
            )
            # インスタンスに対する予測値
            .assign(
                instance=ids_to_compute,
                prediction=self.estimator.predict(self.X[ids_to_compute]),
            )
            # 並べ替え
            .loc[:, ["instance", "prediction"] + self.var_names]
        )

    def plot(self, ylim: list[float] | None = None) -> None:
        """ICEを可視化

        Args:
            ylim: Y軸の範囲。特に指定しなければiceの範囲となる。
        """

        fig, ax = plt.subplots()
        # ICEの線
        sns.lineplot(
            # self.target_var_name,
            # "ice",
            # units="instance",
            data=self.df_ice,
            lw=0.8,
            alpha=0.5,
            estimator=None,
            zorder=1,  # zorderを指定することで、線が背面、点が前面にくるようにする
            ax=ax,
        )
        # インスタンスからの実際の予測値を点でプロットしておく
        sns.scatterplot(
            # self.target_var_name, 
            # "prediction", 
            data=self.df_instance, 
            zorder=2, 
            ax=ax
        )
        ax.set(xlabel=self.target_var_name, ylabel="Prediction", ylim=ylim)
        fig.suptitle(
            f"Individual Conditional Expectation({self.target_var_name})"
        )
        
        fig.show()
        
# ICEのインスタンスを作成

ice = IndividualConditionalExpectation(rf, X_test, ["X0", "X1", "X2"])

# インスタンス0について、X1のICEを計算

ice.individual_conditional_expectation("X1", [0])

# インスタンス0の特徴量と予測値を出力

ice.df_instance

# X2=0だから右肩下がりになるはず

In [ ]:
# インスタンス0のICEを可視化

ice.plot(ylim=(-6, 6))

# 確かに右肩下がり

In [ ]:
# インスタンス1についてX1のICEを計算

ice.individual_conditional_expectation("X1", [1])

# インスタンス1の特徴量と予測値を出力

ice.df_instance

# 今度はX_2=1

In [ ]:
# インスタンス1のICEを可視化

ice.plot(ylim=(-6, 6))

# 見事に右肩上がり

### Conditional Partial Dependence (CPD)

交互作用を疑われる特徴量については条件付けてPDを計算

3変数の場合は以下のようになる

$\widehat{CPD}_{1, 2} (x_1, 0) = \frac {1} {N_0} \sum_{i:x_{i, 2} = 0} \hat{f} (x_{i, 0}, x_1, 0)$

$\widehat{CPD}_{1, 2} (x_1, 1) = \frac {1} {N_1} \sum_{i:x_{i, 2} = 0} \hat{f} (x_{i, 0}, x_1, 1)$

今回の場合は

${\rm CPD}_{1, 2} (x_1, 0) = \mathbb{E} [\hat{f} (X_0, x_1, X_2) | X_2 = 0] = \mathbb{E} [X_0 - 5x_1 + 10x_1X_2 | X_2 = 0] = \mathbb{E} [X_0 | X_2 = 0] - 5x_1 + 10x_1 \mathbb{E} [X_2 | X_2 = 0] = -5x_1$

${\rm CPD}_{1, 2} (x_1, 1) = \mathbb{E} [\hat{f} (X_0, x_1, X_2) | X_2 = 1] = \mathbb{E} [X_0 - 5x_1 + 10x_1X_2 | X_2 = 1] = \mathbb{E} [X_0 | X_2 = 1] - 5x_1 + 10x_1 \mathbb{E} [X_2 | X_2 = 1] = 5x_1$

一般化しておく

${\bf X}_{\backslash \{ j, k \}}$: ${\bf X}$から2つの特徴量$(X_j, X_k)$を取り除いたもの

$p ({\bf x}_{\backslash \{ j, k \} } | x_k)$: $x_k$で条件付けた分布

${\rm CPD}_{j, k} (x_j, x_k) = \mathbb{E} [\hat{f} (x_j, X_k, {\bf X}_{\backslash \{ j, k \}}) | X_k = x_k] = \int \hat{f} (x_j, x_k, {\bf x}_{\backslash \{ j, k \} } | X_k = x_k) p ({\bf x}_{\backslash \{ j, k \} } | x_k) d{\bf x}_{\backslash \{ j, k \} }$

$X_k = x_k$となっているインスタンスの平均を取る

$\widehat{CPD}_{j, k} (x_j, x_k) = \frac {1} {N_k} \sum_{i: x_{i, k}=x_k} \hat{f} (x_j, {\bf x}_{i, \backslash \{ j, k \}})$

In [ ]:
# X2=0のインスタンスに関してX1のPDを計算

pdp = PartialDependence(rf, X_test[X_test[:, 2] == 0], ["X0", "X1", "X2"])
pdp.partial_dependence("X1")

# PDを可視化

pdp.plot(ylim=(-6, 6))

# 右肩下がり

In [ ]:
# X2=1のインスタンスに関してX1のPDを計算

pdp = PartialDependence(rf, X_test[X_test[:, 2] == 1], ["X0", "X1", "X2"])
pdp.partial_dependence("X1")

# PDを可視化

pdp.plot(ylim=(-6, 6))

# 右肩上がり

シミュレーションモデル2

$Y = X_0 + 5X_1 + 10X_1X_2 + \varepsilon$

$X_0 \sim {\rm U} (-1, 1)$

$X_2 \sim {\rm Bernoulli} (0.5)$

$X_1 \sim \left\{
\begin{array}{ll}
{\rm U} (-1, 0.5) & X_2 = 0 \\
{\rm U} (-0.5,1 ) & X_2 = 1
\end{array}
\right.
$

$\varepsilon \sim \mathcal{N} (0, 0.01)$

In [ ]:
# シミュレーションデータの生成

def generate_simulation_data():
    """シミュレーションデータを生成し、訓練データとテストデータに分割"""
    
    # シミュレーションの設定
    N=1000
    
    # X0は一様分布から生成
    x0 = np.random.uniform(-1, 1, N)
    # 二項分布の試行回数を1にすると成功確率0.5のベルヌーイ分布と一致
    x2 = np.random.binomial(1, 0.5, N)
    # X1はX2に依存する形にする
    x1 = np.where(
        x2 == 1, 
        np.random.uniform(-0.5, 1, N), 
        np.random.uniform(-1, 0.5, N)
    )
    # ノイズは正規分布からデータを生成
    epsilon = np.random.normal(0, 0.1, N)
    
    # 特徴量をまとめる
    X = np.column_stack((x0, x1, x2))
    
    # 線形和で目的変数を作成
    y = x0 - 5 * x1 + 10 * x1 * x2 + epsilon
    
    return train_test_split(X, y, test_size=0.2, random_state=42)


X_train, X_test, y_train, y_test = generate_simulation_data()

def plot_scatter(x, y, group, title=None, xlabel=None, ylabel=None):
    """散布図を作成する"""
    
    fig, ax = plt.subplots()
    sns.scatterplot(x=x, y=y, style=group, hue=group, alpha=0.5, ax=ax)
    ax.set(xlabel=xlabel, ylabel=ylabel)
    fig.suptitle(title)

    fig.show()


# X1とYの散布図を作成

plot_scatter(
    X_train[:, 1],
    y_train,
    X_train[:, 2].astype(int),
    title="X1とYの散布図",
    xlabel="X1",
    ylabel="Y",
)

# 上の方に広がったX型

In [ ]:
# ランダムフォレストで予測モデルを構築

rf = RandomForestRegressor(n_jobs=-1, random_state=42).fit(X_train, y_train)

# 予測精度を確認

regression_metrics(rf, X_test, y_test)

# なかなかの高性能予測モデル

In [ ]:
# インスタンス0に関して特徴量X1のICEを計算

ice = IndividualConditionalExpectation(rf, X_test, ["X0", "X1", "X2"])
ice.individual_conditional_expectation("X1", [0])

# インスタンスの特徴量を確認

ice.df_instance

In [ ]:
# ICEを可視化

ice.plot(ylim=(-6, 6))

# 歪んだV字型

### SHAP (SHapley Additive exPlanation)

モデルの予測がなぜその値になったのかを説明する方法

#### SHAPの数式表現

特徴量  ${\bf X} = (X_1, \ldots, X_J)$

学習済みの機械学習モデル $\hat{f} (X)$

インスタンス$i$の特徴量 ${\bf x}_i = (x_{i, 1}, \ldots, x_{i, J})$

インスタンス$i$の特徴量$x_{i, j}$の貢献度 $\phi_{i, j}$

$\hat{f} ({\bf x}_i) - \mathbb{E} [\hat{f} ({\bf X})] = \sum^J_{j=1} \phi_{i, j}$ $\leftarrow$ Additive Feature Attribution Method

$\phi_0 = \mathbb{E} [\hat{f} ({\bf X})]$とおくと(単なるベースライン)

$\hat{f} ({\bf x}_i) = \phi_0 + \sum^J_{j=1} \phi_{i, j}$

各特徴量の貢献度$\phi_{i, j}$に興味あり

#### 貢献度の分解

ex1. 線形回帰モデル

学習済みの線形回帰モデル $\hat{f} ({\bf X}) = \hat{\beta}_0 + \sum^J_{j=1} \hat{\beta}_j X_j$

ベースラインである予測の期待値 $\mathbb{E} [\hat{f} ({\bf X})] = \mathbb{E} [\hat{\beta}_0 + \sum^J_{j=1} \hat{\beta}_j X_j] = \hat{\beta}_0 + \sum^J_{j=1} \hat{\beta}_j \mathbb{E} [X_j]$

インスタンス$i$の予測値と予測の期待値の差分

$\hat{f}({\bf x}_i) - \mathbb{E} [\hat{f}({\bf X})] = (\hat{\beta}_0 + \sum^J_{j=1} \hat{\beta}_j x_{i, j}) - (\hat{\beta}_0 + \sum^J_{j=1} \hat{\beta}_j \mathbb{E} [X_j]) = \sum^J_{j=1} \hat{\beta}_j (x_{i, j} - \mathbb{E} [X_j])$

この$\phi_{i, j} = \hat{\beta}_j (x_{i, j} - \mathbb{E} [X_j])$が特徴量$x_{i, j}$の貢献度

回帰係数$\hat{\beta}_j$が大きいほど貢献度が高くなる

$x_{i, j}$が$\mathbb{E} [X_j]$から乖離するほど貢献度が高くなる

### 協力ゲーム理論とSharpley値

#### アルバイトゲーム (cf. [Okada, 2011])

ex. アルバイトの参加者と報酬の関係

| 参加者 | 報酬 (万円) |
| :---: | :---: |
| Aのみ | 6 |
| Bのみ | 4 |
| Cのみ | 2 |
| AとB | 20 |
| AとC | 15 |
| BとC | 10 |
| 全員 | 24 |

全員が参加した場合に報酬をどのように分配すべきか?

#### 限界貢献度

各人がアルバイトに参加したときの報酬の増加量

ex. Aについて

誰もいない $\rightarrow$ A参加: $6 - 0 = 6$万円

Bのみ $\rightarrow$ Aも参加: $20 - 4 = 16$万円

Cのみ $\rightarrow$ Aも参加: $15 - 2 = 13$万円

AとB $\rightarrow$ Aも参加: $24 - 10 = 14$万円

参加順を考慮した限界貢献度

| 参加順 | Aの限界貢献度 | Bの限界貢献度 | Cの限界貢献度 |
| :---: | :---: | :---: | :---: |
| A$\rightarrow$B$\rightarrow$C | 6 | 14 | 4 |
| A$\rightarrow$C$\rightarrow$B | 6 | 9 | 9 |
| B$\rightarrow$A$\rightarrow$C | 16 | 4 | 4 |
| B$\rightarrow$C$\rightarrow$A | 14 | 4 | 6 |
| C$\rightarrow$A$\rightarrow$B | 13 | 9 | 2 |
| C$\rightarrow$B$\rightarrow$A | 14 | 8 | 2 |

平均的な限界貢献度=Shapley値

A: $(6+6+16+14+13+14) / 6 = 11.5$万円

B: $(14+9+4+4+9+8) / 6 = 8$万円

C: $(4+9+4+6+2+2) / 6 = 4.5$万円

#### Shapley値

$\mathcal{J} = \{ 1, \ldots, J \}$: プレイヤーの集合 人数$=|\mathcal{J}|$

$\mathcal{S}$: $\mathcal{J}$からプレイヤー$j$を除いた集合 (空集合も含む) 集合の数$=|\mathcal{S}|$

$v(\cdot)$: 報酬を表す関数 引数は0人以上のプレイヤーの組

プレイヤー$j$のShapley値

$\phi_j = \frac {1} {|\mathcal{S}|!} \sum_{\mathcal{S} \subseteq \mathcal{J} \backslash \{ j \}} (|\mathcal{S}|! (|\mathcal{J}| - |\mathcal{S}| - 1)!) (v(\mathcal{S} \cup \{ j \}) - v(\mathcal{S}))$

最後の項がプレイヤー$j$が参加したときの限界貢献度

#### SHAPにおけるShapley値

$v(\mathcal{S} \cup \{ j \}) - v(\mathcal{S})$: 特徴量$j$がわかっているときとそうでないときの予測値の差分

ex. 特徴量2個の場合

特徴量 $(X_1, X_2)$

インスタンス$i$での値 $(x_{i, 1}, x_{i, 2})$がわかっている場合

$v(\{ 1, 2 \}) = \hat{f} (x_{i, 1}, x_{i, 2})$

両方の値がわかっていない場合

$v(\emptyset) = \mathbb{E} [\hat{f} (X_1, X_2)]$

$x_{i, 1}$だけがわかっている場合

$v(\{ 1 \}) = \mathbb{E} [\hat{f} (x_{i, 1}, X_2)] = \int \hat{f} (x_{i, 1}, x_2) p (x_2) dx_2$

$x_{i, 1} \rightarrow x_{i, 2}$と値が判明したときの予測値の変化

情報なし$\rightarrow x_{i, 1}$: $\Delta_{i, 1} = \mathbb{E} [\hat{f} (x_{i, 1}, X_2)] - \mathbb{E} [\hat{f} (X_1, X_2)]$

$x_{i, 1} \rightarrow (x_{i, 1}, x_{i, 2})$: $\Delta_{i, 2} = \mathbb{E} [\hat{f} (x_{i, 1}, x_{i, 2})] - \mathbb{E} [\hat{f} (x_{i, 1}, X_2)]$

$x_{i, 2}$が先に判明した場合も含めて平均を取ると

$\phi_{i, 1} = \frac {1} {2} ((\mathbb{E} [\hat{f} (x_{i, 1}, X_2)] - \mathbb{E} [\hat{f} (X_1, X_2)]) + (\mathbb{E} [\hat{f} (x_{i, 1}, x_{i, 2})] - \mathbb{E} [\hat{f} (X_1, x_{i, 2})]))$

$\phi_{i, 2} = \frac {1} {2} ((\mathbb{E} [\hat{f} (x_{i, 1}, x_{i, 2})] - \mathbb{E} [\hat{f} (x_{i, 1}, X_2)]) + (\mathbb{E} [\hat{f} (X_1, x_{i, 2})] - \mathbb{E} [\hat{f} (X_1, X_2)]))$

これがSHAP値

一応合計してみる

$\phi_{i, 1} + \phi_{i, 2} = \mathbb{E} [\hat{f} (x_{i, 1}, x_{i, 2})] - \mathbb{E} [\hat{f} (X_1, X_2)]$

シミュレーションデータ

$Y = X_1 + \varepsilon$

$\begin{pmatrix}
   X_0 \\
   X_1
\end{pmatrix} \sim \mathcal{N} (
\begin{pmatrix}
   0 \\
   0
\end{pmatrix},
\begin{pmatrix}
   1 & 0 \\
   0 & 1
\end{pmatrix}
)$

$\varepsilon \sim \mathcal{N} (0, 0.01)$

In [ ]:
# シミュレーションデータの生成

import sys
import warnings
from dataclasses import dataclass
from typing import Any
from __future__ import annotations

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import japanize_matplotlib

from mli.visualize import get_visualization_setting

np.random.seed(42)
pd.options.display.float_format = "{:.2f}".format
sns.set(**get_visualization_setting())
warnings.simplefilter("ignore")

from sklearn.model_selection import train_test_split

def generate_simulation_data():
    """シミュレーションデータを生成し、訓練データとテストデータに分割する"""
    
    # シミュレーションの設定
    N = 1000
    J = 2
    beta = np.array([0, 1])

    # 特徴量とノイズは正規分布から生成
    X = np.random.normal(0, 1, [N, J])
    e = np.random.normal(0, 0.1, N)
    
    # 線形和で目的変数を作成
    y = X @ beta + e
    
    return train_test_split(X, y, test_size=0.2, random_state=42)
    

# シミュレーションデータを生成

X_train, X_test, y_train, y_test = generate_simulation_data()

from sklearn.ensemble import RandomForestRegressor
from mli.metrics import regression_metrics  # 2.3節で作成した精度評価関数


# ランダムフォレストで予測モデルを構築

rf = RandomForestRegressor(n_estimators=500, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

# 予測精度の評価

regression_metrics(rf, X_test, y_test)

# かなりまともな予測モデル

In [ ]:
# 目的変数と予測値のデータフレームをつくる

df = pd.DataFrame(data=X_test, columns=["X0", "X1"])

# インスタンスごとの予測値

df["y_pred"] = rf.predict(X_test)

# ベースラインとしての予測の平均

df["y_pred_baseline"] = rf.predict(X_test).mean()
df.head()

# この中で予測値が最大なのがインスタンス1
# 予測値-ベースラインの差分=0.63

In [ ]:
# インスタンス1に対する予測値の増え方を確認

x = X_test[1]

# CASE1: X0もX1も分かっていないときの予測値（予測の平均）

E_baseline = rf.predict(X_test).mean()

# CASE2: X0のみが分かっているときの予測値
# 全データのX0の値をインスタンス1のX0の値に置き換えて予測を行って平均を計算

X0 = X_test.copy()
X0[:, 0] = x[0]
E0 = rf.predict(X0).mean()

# CASE3: X1のみが分かっているときの予測値
# 全データのX1の値をインスタンス1のX1の値に置き換えて予測を行って平均を計算

X1 = X_test.copy()
X1[:, 1] = x[1]
E1 = rf.predict(X1).mean()

# CASE4: X0もX1も分かっているときの予測値

E_full = rf.predict(x[np.newaxis, :])[0]

# 結果を出力

print("CASE1: X0もX1も分かっていないときの予測値 -> {0:.2f}".format(E_baseline))
print("CASE2: X0のみが分かっているときの予測値 -> {0:.2f}".format(E0))
print("CASE3: X1のみが分かっているときの予測値 -> {0:.2f}".format(E1))
print("CASE4: X0もX1も分かっているときの予測値 -> {0:.2f}".format(E_full))

In [ ]:
# X0とX1のSHAP値

SHAP0 = ((E0 - E_baseline) + (E_full - E1)) / 2
SHAP1 = ((E1 - E_baseline) + (E_full - E0)) / 2

print("(SHAP0, SHAP1) = {0:.2f}, {1:.2f}".format(SHAP0, SHAP1))

# SHAP0の値が非常に低いのはモデルから見て当然
# 足したら0.63になる (丸めのせいで少しズレている)

In [ ]:
# SHAPを計算するお手製クラス

from scipy.special import factorial
from itertools import combinations

@dataclass
class ShapleyAdditiveExplanations:
    """SHapley Additive exPlanations
    
    Args:
        estimator: 学習済みモデル
        X: SHAPの計算に使う特徴量
        var_names: 特徴量の名前
    """
    
    estimator: Any
    X: np.ndarray
    var_names: list[str]
        
    def __post_init__(self) -> None:
        # ベースラインとしての平均的な予測値
        self.baseline = self.estimator.predict(self.X).mean()

        # 特徴量の総数
        self.J = self.X.shape[1]

        # あり得るすべての特徴量の組み合わせ
        self.subsets = [
            s
            for j in range(self.J + 1)
            for s in combinations(range(self.J), j)
        ]

    def _get_expected_value(self, subset: tuple[int, ...]) -> np.ndarray:
        """特徴量の組み合わせを指定するとその特徴量が場合の予測値を計算

        Args:
            subset: 特徴量の組み合わせ
        """
        
        _X = self.X.copy()  # 元のデータが上書きされないように

        # 特徴量がある場合は上書き。なければそのまま。
        if subset is not None:
            # 元がtupleなのでリストにしないとインデックスとして使えない
            _s = list(subset)
            _X[:, _s] = _X[self.i, _s]

        return self.estimator.predict(_X).mean()

    def _calc_weighted_marginal_contribution(
        self,
        j: int,
        s_union_j: tuple[int, ...]
    ) -> float:
        """限界貢献度x組み合わせ出現回数を求める

        Args:
            j: 限界貢献度を計算したい特徴量のインデックス
            s_union_j: jを含む特徴量の組み合わせ
        """
        
        # 特徴量jがない場合の組み合わせ
        s = tuple(sorted(set(s_union_j) - set([j])))

        # 組み合わせの数
        S = len(s)

        # 組み合わせの出現回数
        # ここでfactorial(self.J)で割ってしまうと丸め誤差が出てるので、あとで割る
        weight = factorial(S) * factorial(self.J - S - 1)

        # 限界貢献度
        marginal_contribution = (
            self.expected_values[s_union_j] - self.expected_values[s]
        )

        return weight * marginal_contribution

    def shapley_additive_explanations(self, id_to_compute: int) -> None:
        """SHAP値を求める

        Args:
            id_to_compute: SHAPを計算したいインスタンス
        """

        # SHAPを計算したいインスタンス
        self.i = id_to_compute

        # すべての組み合わせに対して予測値を計算
        # 先に計算しておくことで同じ予測を繰り返さずに済む
        self.expected_values = {
            s: self._get_expected_value(s) for s in self.subsets
        }

        # ひとつひとつの特徴量に対するSHAP値を計算
        shap_values = np.zeros(self.J)
        for j in range(self.J):
            # 限界貢献度の加重平均を求める
            # 特徴量jが含まれる組み合わせを全部もってきて
            # 特徴量jがない場合の予測値との差分を見る
            shap_values[j] = np.sum([
                self._calc_weighted_marginal_contribution(j, s_union_j)
                for s_union_j in self.subsets
                if j in s_union_j
            ]) / factorial(self.J)
        
        # データフレームとしてまとめる
        self.df_shap = pd.DataFrame(
            data={
                "var_name": self.var_names,
                "feature_value": self.X[id_to_compute],
                "shap_value": shap_values,
            }
        )

    def plot(self) -> None:
        """SHAPを可視化"""
        
        # 下のデータフレームを書き換えないようコピー
        df = self.df_shap.copy()
        
        # グラフ用のラベルを作成
        df['label'] = [
            f"{x} = {y:.2f}" for x, y in zip(df.var_name, df.feature_value)
        ]
        
        # SHAP値が高い順に並べ替え
        df = df.sort_values("shap_value").reset_index(drop=True)
        
        # 全特徴量の値がときの予測値
        predicted_value = self.expected_values[self.subsets[-1]]
        
        # 棒グラフを可視化
        fig, ax = plt.subplots()
        ax.barh(df.label, df.shap_value)
        ax.set(xlabel="SHAP値", ylabel=None)
        fig.suptitle(f"SHAP値 \n(Baseline: {self.baseline:.2f}, Prediction: {predicted_value:.2f}, Difference: {predicted_value - self.baseline:.2f})")

        fig.show()
        
# SHAPのインスタンスを作成

shap = ShapleyAdditiveExplanations(rf, X_test, ["X0", "X1"])

# インスタンス1に対してSHAP値を計算

shap.shapley_additive_explanations(id_to_compute=1)

# SHAP値を出力

shap.df_shap

In [ ]:
# SHAP値を計算

shap.shapley_additive_explanations(id_to_compute=1)

# SHAP値を可視化

shap.plot()